# Talking Characters — Google Colab Version

Colab-friendly version of the talking-characters pipeline.
End-to-end: YouTube CSV → cleaned single-speaker talking-head clips.

**Differences from the HPC version:**
- 1 GPU instead of 4 (Ray actor counts are scaled down)
- No PBS / HPC offline-mode (compute has internet here)
- `wandb` is optional — left disabled by default
- Outputs go to `/content/talking_characters_data` (optionally mount Drive)

**Required Colab runtime:** `Runtime → Change runtime type → GPU` (T4 is fine; A100 is fastest).

**Before running:** the user is expected to have already `git clone`'d the repo into `/content/talking-characters` (or wherever). The first cell verifies that.

## 0. Verify repo + GPU

If you have not cloned the repo yet, run this in a cell first:
```
!git clone https://github.com/yashjain14/talking-characters.git /content/talking-characters
%cd /content/talking-characters
```

In [ ]:
import os, sys, subprocess

REPO_DIR = "/content/talking-characters"
if not os.path.isdir(REPO_DIR):
    raise SystemExit(f"Repo not found at {REPO_DIR}. Run `!git clone <repo> {REPO_DIR}` first.")

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
print("files:", sorted(os.listdir())[:20])

# Confirm GPU is attached
subprocess.run(["nvidia-smi", "-L"], check=False)

## 1. Install dependencies

Mirrors `setup_env.sh` but adapted for the Colab base image (Python 3.10/3.11, torch already preinstalled).
We pin `numpy<2` and `opencv-python-headless<4.11` per the project's ABI constraints.

Takes ~3–5 min.

In [ ]:
# System deps: ffmpeg for audio extract + clip encode
!apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null
!ffmpeg -version | head -n 1

In [ ]:
# Pin numpy<2 first — many deps build against the 1.x ABI
!pip install -q "numpy<2"

# Video decode (CPU PyAV — pipeline avoids PyNvVideoCodec for fractional-GPU actors)
!pip install -q av

# OpenCV: must stay <4.11 because 4.11 ships against NumPy 2 ABI
!pip install -q "opencv-python-headless<4.11"

# YouTube download + audio
!pip install -q yt-dlp soundfile python_speech_features

# Face detection + tracking
!pip install -q insightface onnxruntime-gpu boxmot

# Active speaker detection (try both; either will fail gracefully if not on PyPI)
!pip install -q loconet light-asd || echo "(loconet/light-asd from PyPI may need a git+ install if these names aren't published)"

# Ray + Prefect + (optional) wandb
!pip install -q "ray[default]" prefect wandb

print("\nInstall done.")

In [ ]:
# Sanity check imports and CUDA
import numpy, cv2, torch, av, insightface
print("numpy           ", numpy.__version__)
print("opencv          ", cv2.__version__)
print("torch           ", torch.__version__, "cuda:", torch.cuda.is_available())
print("av              ", av.__version__)
print("insightface     ", insightface.__version__)
if torch.cuda.is_available():
    print("gpu             ", torch.cuda.get_device_name(0))

## 2. Configure paths + (optional) Drive mount

By default, outputs go to `/content/talking_characters_data`, which is **wiped when the Colab runtime ends**. To persist clips, mount Drive and set `SCRATCH_TC` under it.

In [ ]:
USE_DRIVE = False  # set True to persist outputs across sessions

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SCRATCH_TC = "/content/drive/MyDrive/talking_characters_data"
else:
    SCRATCH_TC = "/content/talking_characters_data"

os.environ["SCRATCH_TC"] = SCRATCH_TC
os.makedirs(SCRATCH_TC, exist_ok=True)
print("SCRATCH_TC =", SCRATCH_TC)

# Disable wandb by default in Colab (no API key). Set WANDB_API_KEY env to enable.
if not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_MODE"] = "disabled"
    print("wandb: disabled (set WANDB_API_KEY to enable)")

## 3. Prefetch model weights

Downloads InsightFace `buffalo_sc` (SCRFD-10GF) and LoCoNet / Light-ASD weights into `~/.cache/talking_characters`.

(On Colab compute has internet, so you could technically skip this — but the same script that runs in the pipeline expects the files at these cache paths, so running it once is the safe path.)

In [ ]:
!python prefetch_models_talking_characters.py

## 4. Reduce Ray actor counts for a single GPU

The original pipeline targets 4× A100s (16 detect actors, 8 ASD actors). On Colab's single GPU we shrink to 4 detect actors (0.25 GPU each) and 2 ASD actors (0.5 GPU each). The constants live in the stage scripts; we patch them in-place using `sed`. (Each run is idempotent — re-running this cell is fine.)

Skip this cell if you want to keep the original concurrency.

In [ ]:
# Nothing to change — ACTORS_PER_GPU is per-GPU, and we pass num_gpus=1 below.
# detect_track.py:  ACTORS_PER_GPU = 4  → 4 actors  (was 16 across 4 GPUs)
# active_speaker.py: ACTORS_PER_GPU = 2 → 2 actors  (was  8 across 4 GPUs)
# segment_export uses CPU only; we pass num_workers=4.
!grep -n ACTORS_PER_GPU detect_track.py active_speaker.py

## 5. Provide your input CSV

Either edit `videos.csv` in the repo, or write your own list below.

In [ ]:
CSV_PATH = "/content/talking-characters/videos.csv"

# Example: overwrite with your own list. Comment this block out to keep the repo default.
with open(CSV_PATH, "w") as f:
    f.write("url,label\n")
    f.write("https://www.youtube.com/watch?v=57lDpTwiW6g,demo\n")

!cat {CSV_PATH}

## 6. Run the pipeline

Use `--num_gpus 1` on Colab. `--asd_model light_asd` is faster if you don't need LoCoNet's multi-speaker accuracy.

In [ ]:
!python dag_talking_characters.py \
    --csv {CSV_PATH} \
    --num_gpus 1 \
    --ingest_workers 2 \
    --asd_model loconet

### Resume from a stage

Re-run cells without re-downloading or re-detecting:
```
!python dag_talking_characters.py --csv {CSV_PATH} --num_gpus 1 --from_stage detect_track
!python dag_talking_characters.py --csv {CSV_PATH} --num_gpus 1 --from_stage active_speaker
!python dag_talking_characters.py --csv {CSV_PATH} --num_gpus 1 --from_stage segment_export
```

## 7. Inspect outputs

In [ ]:
from pathlib import Path
import json

root = Path(SCRATCH_TC)
for sub in ["raw_videos", "tracks", "asd", "clips"]:
    p = root / sub
    if p.exists():
        files = list(p.rglob("*"))
        print(f"{sub:12s}  {len(files):4d} files  → {p}")
    else:
        print(f"{sub:12s}  (missing)")

manifest = root / "clips" / "segments.json"
if manifest.exists():
    clips = json.loads(manifest.read_text())
    print(f"\n{len(clips)} clips in segments.json")
    for c in clips[:5]:
        print(f"  {c['duration_s']:.1f}s  active={c['active_ratio']:.2f}  {c['clip_path']}")

In [ ]:
# Preview the first exported clip inline
from pathlib import Path
from IPython.display import Video, display

mp4s = sorted(Path(SCRATCH_TC, "clips").rglob("*.mp4"))
if not mp4s:
    print("No clips yet.")
else:
    print("First clip:", mp4s[0])
    display(Video(str(mp4s[0]), embed=True, width=480))

## Troubleshooting

- **`No module named 'loconet'` / `light_asd`** — these aren't on PyPI under those exact names in all versions; if the install line failed silently, install from source: `!pip install git+https://github.com/<author>/loconet` and likewise for Light-ASD. Or use `--asd_model light_asd` if you only got one of them.
- **`onnxruntime-gpu` CUDA mismatch** — Colab's CUDA may not match the wheel. Fall back to CPU detection: `!pip install onnxruntime` and edit `detect_track.py` `providers=["CPUExecutionProvider"]`. Slower but works.
- **Out of disk** — the `/content` disk is ~80GB. For larger CSVs, mount Drive (set `USE_DRIVE = True` in step 2).
- **Ray serialization warnings about `numpy>=2`** — make sure step 1's `numpy<2` pin took effect (`import numpy; print(numpy.__version__)` should print `1.x`).
- **`CUDA_ERROR_CONTEXT_IS_DESTROYED`** — don't switch to PyNvVideoCodec decode; the pipeline deliberately uses PyAV CPU decode because fractional-GPU actors share a single physical GPU.